
# Data Loading

We load the pre-processed training (2015–2018) and blind test (2019) datasets. These datasets have finished our feature engineering pipeline.

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
from pyspark.sql.functions import regexp_replace, mean, stddev, log1p, concat_ws
import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time

spark.conf.set("spark.sql.ansi.enabled", "false")

TEAM_DIR = "dbfs:/student-groups/Group_3_2"
otpw_train_df = spark.read.parquet(f"{TEAM_DIR}/041026_otpw_train_df.parquet")
otpw_test_df = spark.read.parquet(f"{TEAM_DIR}/041026_otpw_test_df.parquet")

print(f"Train (2015-2018): {otpw_train_df.count():,} rows x {len(otpw_train_df.columns)} columns")
print(f"Test  (2019):      {otpw_test_df.count():,} rows x {len(otpw_test_df.columns)} columns")

In [0]:
label_col = "DEP_DEL15"

# Verify class balance
train_dist = otpw_train_df.groupBy(label_col).count().toPandas()
train_dist["pct"] = (train_dist["count"] / train_dist["count"].sum() * 100).round(2)
print(f"\nTrain class distribution:\n{train_dist}")

# Modeling Pipeline

We apply the modeling pipeline to the 4-year training dataset (2015 - 2018) with engineered features, evaluating Logistic Regression, Random Forest, and Gradient Boosted Trees. The 2019 dataset is held out as a blind test set that is never consulted during training or cross-validation.

### Build Spark ML Pipeline

The pipeline applies `StringIndexer`, `OneHotEncoder` for each categorical feature, then assembles all numeric and encoded categorical features into a single vector using `VectorAssembler`.

In [0]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Shared categorical features (used by LR and tree-based models)
categorical_features = [
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR",
    "VisibilityCat",
]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

# 1. Logistic Regression Pipeline (Baseline)
#    Replaces delayed_flights_2_4hr_before with log_total_flights_2_4hr_before
#    to avoid multicollinearity (count = pct x total)
lr_numeric_features = [
    # Weather
    "HourlyDryBulbTemperature_stdized",
    "HourlyWindSpeed_stdized",
    "PrecipBinary",
    "HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (log of total flights replaces raw count)
    "delay_pct_flights_2_4hr_before", "log_total_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pr_weighted", "origin_pr_unweighted",
    "dest_pr_weighted", "dest_pr_unweighted",
]

lr_assembler_inputs = lr_numeric_features + [f"{c}_ohe" for c in categorical_features]
lr_assembler = VectorAssembler(inputCols=lr_assembler_inputs, outputCol="features", handleInvalid="skip")
lr_pipeline_stages = indexers + encoders + [lr_assembler]

# 2. Tree-based Pipeline (RF, GBT)
tree_numeric_features = [
    # Weather
    "HourlyDryBulbTemperature_stdized",
    "HourlyWindSpeed_stdized",
    "PrecipBinary",
    "HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series (short-term airport congestion)
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (airport importance)
    "origin_pr_weighted", "origin_pr_unweighted",
    "dest_pr_weighted", "dest_pr_unweighted",
]

tree_assembler_inputs = tree_numeric_features + [f"{c}_ohe" for c in categorical_features]
tree_assembler = VectorAssembler(inputCols=tree_assembler_inputs, outputCol="features", handleInvalid="skip")
tree_pipeline_stages = indexers + encoders + [tree_assembler]

# 3. MLP Pipeline (reduced feature set for neural network)
#    Drops ORIGIN OHE (high cardinality might high dimentional sparse inputs for MLP), uses PageRank instead
mlp_numeric_features = [
    # Weather
    "HourlyDryBulbTemperature_stdized", "HourlyWindSpeed_stdized",
    "PrecipBinary", "HourlyRelativeHumidity_stdized",
    # Schedule
    "DISTANCE", "IsWeekend",
    # Time-series
    "delayed_flights_2_4hr_before", "delay_pct_flights_2_4hr_before",
    # Graph (replaces ORIGIN/DEST)
    "origin_pr_weighted", "origin_pr_unweighted",
    "dest_pr_weighted", "dest_pr_unweighted",
]

mlp_categorical_features = [
    "OP_UNIQUE_CARRIER", "DAY_OF_WEEK", "MONTH", "DEP_HOUR", "VisibilityCat",
]

mlp_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_mlp_idx", handleInvalid="keep")
    for c in mlp_categorical_features
]
mlp_encoders = [
    OneHotEncoder(inputCol=f"{c}_mlp_idx", outputCol=f"{c}_mlp_ohe")
    for c in mlp_categorical_features
]
mlp_assembler_inputs = mlp_numeric_features + [f"{c}_mlp_ohe" for c in mlp_categorical_features]
mlp_assembler = VectorAssembler(inputCols=mlp_assembler_inputs, outputCol="features", handleInvalid="skip")
mlp_pipeline_stages = mlp_indexers + mlp_encoders + [mlp_assembler]

# Determine MLP input dimension from training data
mlp_prep = Pipeline(stages=mlp_pipeline_stages).fit(otpw_train_df)
sample_vec = mlp_prep.transform(otpw_train_df.limit(1)).select("features").head()[0]
mlp_input_dim = len(sample_vec)

# Summary
print(f"LR:    {len(lr_numeric_features)} numeric + {len(categorical_features)} categorical = {len(lr_assembler_inputs)} assembler inputs")
print(f"Tree:  {len(tree_numeric_features)} numeric + {len(categorical_features)} categorical = {len(tree_assembler_inputs)} assembler inputs")
print(f"MLP:   {len(mlp_numeric_features)} numeric + {len(mlp_categorical_features)} categorical = {mlp_input_dim} input dim (after OHE)")


### Evaluation Metrics

For airlines, the cost of a **false negative** (predicting on-time when the flight is actually delayed) is significantly higher than a **false positive** (predicting delayed when the flight departs on time):

- **False Negative (missed delay):** The airline fails to prepare: crew scheduling is disrupted last-minute, gate reassignments increases, passengers miss connections, and a single missed delay can trigger a chain of downstream issues. The financial and reputational cost is high.
- **False Positive (false alarm):** The airline pre-allocates standby resources that weren't needed: extra crew on standby, a backup gate reserved, passengers notified of a potential delay. The cost is moderate (some wasted resources) but overall harmless.

This motivates our choice of **F2-score** as the primary evaluation metric. F2 combines precision and recall into a single score but places twice the weight on recall, penalizing missed delays more heavily than false alarms. We use F2 rather than raw recall because optimizing recall alone is trivial (predict every flight as delayed), while F2 ensures the model maintains reasonable precision.

| Metric  | Role | Why |
|--------|------|-----|
| **F2**  | Primary | Balances precision and recall with 2x weight on recall, preventing missed delays while keeping predictions actionable |
| **Recall** | Supporting | Measures raw delay detection rate; F2's recall-weighted design ensures this stays high |
| **Precision** | Supporting | Ensures predicted delays are actionable, not just noise |
| **AUC-PR**  | Supporting | Captures the full precision-recall trade-off across all thresholds |

In [0]:
def evaluate_model(predictions, label_col="DEP_DEL15"):
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")

    tp = predictions.filter((col("prediction") == 1) & (col(label_col) == 1)).count()
    fp = predictions.filter((col("prediction") == 1) & (col(label_col) == 0)).count()
    fn = predictions.filter((col("prediction") == 0) & (col(label_col) == 1)).count()

    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f2 = round(5 * precision * recall / (4 * precision + recall), 4) if (precision + recall) > 0 else 0.0

    return {
        "AUC-PR": round(evaluator_pr.evaluate(predictions), 4),
        "Precision": precision,
        "Recall": recall,
        "F2": f2,
    }

### Modeling Approaches

We use **Logistic Regression** as the baseline. We then compare against tree-based models and a neural network:

- **Random Forest:** Handles non-linear feature interactions without explicit engineering. Robust to outliers and noisy features.
- **Gradient Boosted Trees:** Sequentially builds trees to correct previous errors, typically achieving higher predictive performance than Random Forest. More prone to overfitting.
- **Multilayer Perceptron (MLP):** A feed-forward neural network that learns non-linear decision boundaries through hidden layers. We use a reduced feature set for MLP that replaces high-cardinality categorical features (`ORIGIN`) with continuous graph-based representations (PageRank) to avoid the dimensionality issues caused by one-hot encoding in dense neural networks.

In [0]:
def run_experiment(model, pipeline_stages, train_df, test_df, model_name, dataset_name):
    full_pipeline = Pipeline(stages=pipeline_stages + [model])

    start = time.time()
    fitted = full_pipeline.fit(train_df)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_df)
    test_preds = fitted.transform(test_df)

    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    print(f"\n============================================================")
    print(f"{model_name} on {dataset_name}")
    print(f"Training time: {train_time}s")
    print(f"\n============================================================")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"------------------------------------------")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Train Time (s)": train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }, fitted

### Class Imbalance Handling

The training data exhibits a natural class imbalance of approximately 80% on-time flights and 20% delayed flights. Without correction, models trained on this distribution would be biased toward predicting the majority class (on-time), resulting in high accuracy but poor recall for delays.

We apply **random undersampling** to the majority class (on-time flights) to match the minority class count, creating a 50/50 balanced training set. This forces models to learn delay patterns rather than defaulting to the majority class. Undersampling is applied **only to training data**, and all test and validation sets retain their natural class distribution to accurately simulate real-world conditions. During cross-validation, undersampling is performed independently within each fold after the temporal split.

In [0]:
def undersample_majority(df, label_col="DEP_DEL15", seed=42):
    """Undersample the majority class (label=0) to match the minority class count."""
    major_df = df.filter(col(label_col) == 0)
    minor_df = df.filter(col(label_col) == 1)
    major_count = major_df.count()
    minor_count = minor_df.count()
    undersample_ratio = minor_count / major_count
    major_undersampled = major_df.sample(withReplacement=False, fraction=undersample_ratio, seed=seed)
    return major_undersampled.unionAll(minor_df)

## Cross-Validation on Training Data (2015 - 2018)

We use **yearly sliding-window cross-validation** on the 4-year training set with a fixed 2-year training window:

| Fold | Train | Test |
|------|-------|------|
| 1 | 2015 - 2016 | 2017 |
| 2 | 2016 - 2017 | 2018 |

This sliding window approach keeps the training size consistent across folds. Each fold trains on a full year data which captures the full holiday and seasonal pattern, and tests on the next year.

### Leakage Prevention

To ensure no future information leaks into training, we enforce the following safeguards in every fold:

1. **Temporal separation:** Each fold trains only on past years and tests on a future year. No random shuffling across time.
2. **Undersampling on train only:** Majority-class downsampling is applied only to the training fold. Testing data does not suppose to know the delay result so it should not be undersampled.
3. **No target leakage:** We predict `DEP_DEL15` (binary delay indicator) using only features available before departure: weather observations, schedule metadata, and airport/carrier identifiers. No post-departure columns (actual delay minutes, arrival info) are included.

### Overfitting Detection

We report **both train and test F2** for every fold. A large gap (train >> test) signals overfitting, the model memorizes training data but fails to generalize, in contract, smaller gap indicates better generalization.

In [0]:
cv_folds = [
    {"train_start": 2015, "train_end": 2016, "test_start": 2017, "test_end": 2017, "label": "Train 2015-2016, Test 2017"},
    {"train_start": 2016, "train_end": 2017, "test_start": 2018, "test_end": 2018, "label": "Train 2016-2017, Test 2018"},
]

def rolling_cv(df, model_fn, pipeline_stages, folds, label_col="DEP_DEL15"):
    """Rolling cross-validation with yearly folds and majority undersampling.
    Reports both train and test metrics per fold for overfitting analysis."""
    fold_results = []
    fitted_models = []

    for i, fold in enumerate(folds, 1):
        train_fold = df.filter(
            (col("YEAR").cast("int") >= fold["train_start"]) &
            (col("YEAR").cast("int") <= fold["train_end"])
        )
        test_fold = df.filter(
            (col("YEAR").cast("int") >= fold["test_start"]) &
            (col("YEAR").cast("int") <= fold["test_end"])
        )

        test_count = test_fold.count()
        if test_count == 0:
            continue

        # Undersample majority class in the training fold
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)

        model = model_fn()
        full_pipeline = Pipeline(stages=pipeline_stages + [model])

        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        fitted_models.append(fitted)

        # Evaluate on both train and test
        train_preds = fitted.transform(train_balanced)
        test_preds = fitted.transform(test_fold)
        train_metrics = evaluate_model(train_preds)
        test_metrics = evaluate_model(test_preds)

        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time,
            "Train_F2": train_metrics["F2"], "Test_F2": test_metrics["F2"],
            "Train_Recall": train_metrics["Recall"], "Test_Recall": test_metrics["Recall"],
            "Train_Precision": train_metrics["Precision"], "Test_Precision": test_metrics["Precision"],
            "Train_AUC-PR": train_metrics["AUC-PR"], "Test_AUC-PR": test_metrics["AUC-PR"],
        })

        print(f"\nFold {i}: {fold['label']} ({train_time}s)")
        print(f"  {'Metric':<12} {'Train':>8} {'Test':>8} {'Gap':>8}")
        print(f"  --------------------------------------")
        for m in ["F2", "Recall", "Precision", "AUC-PR"]:
            tr = train_metrics[m]
            te = test_metrics[m]
            print(f"  {m:<12} {tr:>8.4f} {te:>8.4f} {tr - te:>+8.4f}")

    fold_df = pd.DataFrame(fold_results)
    return fold_df, fitted_models